In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
# from helper import load_env
# load_env()
from pydantic import BaseModel, Field
from typing import List, Dict, Type
from typing import List, Optional
import os
import yaml

In [ ]:
import os, json, time, gc
import logging 

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, Image, Video
from tqdm import tqdm
from openai import OpenAI, AsyncOpenAI
from openai.types.chat import (ChatCompletion, 
                               ChatCompletionChunk,
                               ChatCompletionContentPartTextParam, 
                               ChatCompletionContentPartImageParam,
                               ChatCompletionStreamOptionsParam)
import asyncio
import aiohttp
import pandas as pd
import re


import base64
from PIL import Image
import io

#fix bug with aysncio and jupyter
import nest_asyncio # for langchain async 
nest_asyncio.apply()

In [ ]:
import litellm
from litellm import acompletion, completion

### Test LM Studio Connection By Openai API

In [ ]:
LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
api_key= "lm-studio"

In [ ]:
client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=api_key)

# Replace with the exact model name running in LM Studio
model_name = "qwen3.6-35b-a3b-mtp"  #"google/gemma-4-12b" 

In [ ]:
ret = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a helpful AI coding assistant."},
        {"role": "user", "content": "Explain how to check memory usage in a Jupyter notebook."}
    ],
    temperature=0.7,
)

Markdown(ret.choices[0].message.content)

## Test LM studio LLM Connection by liteLLM API

In [ ]:
# Markdown(completion.choices[0].message.content)

In [ ]:
# 3. Define the async function
async def get_chat_completion(
    api_base,
    api_key, 
    model_name = "openai/local-model",
    system_prompt= "You are a helpful assistant.",
    user_prompt="",
    temperature= 0.7,
    max_tokens=4096):
    
    response = await acompletion(
        model=model_name,  # Prefix with 'openai/' so LiteLLM uses the OpenAI structure
        api_base=api_base,
        api_key=api_key,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response

In [ ]:
%%time
# 4. Execute the async function directly in Jupyter
response = asyncio.run(get_chat_completion(api_base=LM_STUDIO_BASE_URL, 
                                           api_key=api_key,
                                           model_name="openai/local-model",
                                            user_prompt="What is LLM?"))



In [ ]:
Markdown(response.choices[0].message.content)

## Concurrent Version 

In [ ]:
import os
import pandas as pd
import asyncio
from tqdm.asyncio import tqdm_asyncio
from litellm import acompletion
import time

In [ ]:
LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
api_key = "lm-studio"
MAX_CONCURRENT = 2
DELAY = 1        # small delay between batches (optional)
BATCH_SIZE = 20 #50      # process in batches for safer saving (number of row)
# set 
semaphore = asyncio.Semaphore(MAX_CONCURRENT)

In [ ]:
# ====================== ASYNC GENERATE FUNCTION ======================
async def async_generate_cot_data(prompt: str, answer: str) -> str:
    system_prompt = """You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles.
Think step by step inside <think> </think> tags.
Focus only on explaining how to discover the hidden rule.
Do NOT output the final answer yourself."""

    user_message = f"""Puzzle:
{prompt}
Correct Answer: {answer}
Please think step by step inside <think> tags about how to discover the transformation rule."""

    async with semaphore:
        try:
            response = await acompletion(
                model="openai/local-model",
                api_base=LM_STUDIO_BASE_URL,
                api_key=api_key,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.3,
                max_tokens=1600,
                timeout=180
            )

            message = response.choices[0].message
            reasoning = getattr(message, "reasoning_content", "") or ""
            content = message.content or ""

            if reasoning:
                thinking_part = f"<think>\n{reasoning.strip()}\n</think>"
            else:
                thinking_part = f"<think>\n{content.strip()}\n</think>"

            return f"{thinking_part}\n\\boxed{{{answer}}}"

        except Exception as e:
            print(f"Error: {e}")
            return f"<think>\nUnable to generate reasoning.\n</think>\n\\boxed{{{answer}}}"

In [31]:
async def generate_cot_with_resume():
    # === Resume Logic ===
    '''
    for concurrent version
    '''
    if os.path.exists(outputFile):
        print("Found existing train_cot.csv → Resuming...")
        trainDF = pd.read_csv(outputFile)
    else:
        print("No existing file found. Starting from train.csv...")
        trainDF = pd.read_csv(trainFile)
        if "cot_reasoning" not in trainDF.columns:
            trainDF["cot_reasoning"] = ""

    # Count remaining rows
    remaining_mask = trainDF["cot_reasoning"].isna() | (trainDF["cot_reasoning"] == "")
    remaining = remaining_mask.sum()

    print(f"Total rows: {len(trainDF)}")
    print(f"Rows already processed: {len(trainDF) - remaining}")
    print(f"Rows left to process: {remaining}\n")

    if remaining == 0:
        print("✅ All rows already have CoT reasoning. Nothing to do.")
        return

    # Get indices that need processing
    indices_to_process = trainDF[remaining_mask].index.tolist()
    print(f"Starting CoT generation with {MAX_CONCURRENT} concurrent requests...\n")

    processed_count = 0

    # Process in batches for safer saving
    for start in range(0, len(indices_to_process), BATCH_SIZE):
        batch_indices = indices_to_process[start : start + BATCH_SIZE]
        batch_tasks = []

        for idx in batch_indices:
            prompt = trainDF.loc[idx, "prompt"]
            answer = str(trainDF.loc[idx, "answer"]).strip()
            task = async_generate_cot_data(prompt, answer)
            batch_tasks.append((idx, task))

        # Run batch concurrently
        results = await tqdm_asyncio.gather(
            *[task for _, task in batch_tasks],
            desc=f"Batch {start // BATCH_SIZE + 1}"
        )

        # Update dataframe
        for (idx, _), result in zip(batch_tasks, results):
            trainDF.loc[idx, "cot_reasoning"] = result
            processed_count += 1

        # Save progress after each batch
        trainDF.to_csv(outputFile, index=False)
        print(f"Saved progress. Processed {processed_count} / {remaining} new rows so far.")

        # Optional small delay between batches
        await asyncio.sleep(DELAY)

    print(f"\n✅ Finished! Processed {processed_count} new rows.")
    print(f"File saved to: {outputFile}")



In [ ]:
# %%time
# asyncio.run(generate_cot_with_resume())

## Generate COT Data single call version

In [32]:
def generate_cot_data(prompt: str, answer: str) -> str:
    """Synchronous version (for easier use in loops)"""
    system_prompt = """You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles.

Think step by step inside <think> </think> tags.
Focus only on explaining how to discover the hidden rule.
Do NOT output the final answer yourself."""

    user_message = f"""Puzzle:
{prompt}

Correct Answer: {answer}

Please think step by step inside <think> tags about how to discover the transformation rule."""

    try:
        response = asyncio.run (acompletion(
            model="openai/local-model",  # Prefix with 'openai/' so LiteLLM uses the OpenAI structure
            api_base=LM_STUDIO_BASE_URL,
            api_key=api_key,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.3,
            max_tokens=1600,
            timeout=180
            # reasoning_effort="medium"
        ))
        message = response.choices[0].message
        reasoning = getattr(message, "reasoning_content", "") or ""
        content = message.content or ""

        # Combine reasoning
        if reasoning:
            thinking_part = f"<think>\n{reasoning.strip()}\n</think>"
        else:
            thinking_part = f"<think>\n{content.strip()}\n</think>"

        # === Hardcode the final answer (Most Reliable) ===
        final_output = f"{thinking_part}\n\\boxed{{{answer}}}"

        return final_output
        
    except Exception as e:
        print(f"Error generating CoT for prompt: {e}")
        # Fallback: still return something usable
        return f"<think>\nUnable to generate reasoning.\n</think>\n\\boxed{{{answer}}}"
                            

In [45]:
def generate_cot_data2(prompt: str, answer: str) -> str:
    """Generate high-quality Chain-of-Thought reasoning for puzzle transformation rules."""
    
    system_prompt = """You are an expert puzzle solver specializing in discovering hidden transformation rules in Alice's Wonderland puzzles.

Your task is to carefully analyze the given examples and figure out the secret rule that transforms the input into the output.

Guidelines:
- Think step by step inside <think> </think> tags.
- Focus on identifying the underlying transformation pattern (e.g., bit manipulation, substitution cipher, mathematical formula, string operation, etc.).
- Explain your reasoning clearly: observe the examples, form a hypothesis about the rule, and verify it.
- Do NOT output the final answer yourself. The final answer will be added separately."""

    user_message = f"""Here is a puzzle with several input → output examples. A secret transformation rule is being applied.

{prompt}

The correct output for the last input is: {answer}

Please analyze the examples carefully and think step by step about what the hidden transformation rule might be.

Write your reasoning inside <think> </think> tags. Focus on discovering the pattern."""

    try:
        response = asyncio.run(acompletion(
            model="openai/local-model",
            api_base=LM_STUDIO_BASE_URL,
            api_key=api_key,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.4,           # Slightly higher for more creative reasoning
            max_tokens=1800,
            timeout=180
        ))

        message = response.choices[0].message
        reasoning = getattr(message, "reasoning_content", "") or ""
        content = message.content or ""

        # Combine reasoning into <think> tags
        if reasoning:
            thinking_part = f"<think>\n{reasoning.strip()}\n</think>"
        else:
            thinking_part = f"<think>\n{content.strip()}\n</think>"

        # Hardcode the final answer (most reliable)
        final_output = f"{thinking_part}\n\\boxed{{{answer}}}"
        return final_output

    except Exception as e:
        print(f"Error generating CoT: {e}")
        return f"<think>\nUnable to generate reasoning.\n</think>\n\\boxed{{{answer}}}"

In [54]:
testFile ="../src/Dataset/test.csv"
trainFile = "../src/Dataset/train.csv"
cotFile = "train_cot.csv"


In [55]:
# trainDF = pd.read_csv(trainFile)
# trainDF

In [56]:
# # Add new column for CoT reasoning
# if "cot_reasoning" not in trainDF.columns:
#     trainDF["cot_reasoning"] = ""

In [57]:
# trainDF

In [58]:
outputFile = "train_cot2.csv"               # output file with CoT
# MODEL = "gpt-4o"                         # or "claude-3-5-sonnet-20241022"
DELAY = 0.1                               # seconds between API calls (adjust based on rate limit)

In [59]:
print("Checking for existing train_cot.csv...")

if os.path.exists(outputFile):
    print("Found existing train_cot.csv → Resuming...")
    trainDF = pd.read_csv(outputFile)
else:
    print("No existing file found. Starting from train.csv...")
    trainDF = pd.read_csv(trainFile)
    if "cot_reasoning" not in trainDF.columns:
        trainDF["cot_reasoning"] = ""

# Count how many rows still need processing
remaining = trainDF["cot_reasoning"].isna().sum() + (trainDF["cot_reasoning"] == "").sum()
print(f"Total rows: {len(trainDF)}")
print(f"Rows already processed: {len(trainDF) - remaining}")
print(f"Rows left to process: {remaining}\n")

Checking for existing train_cot.csv...
Found existing train_cot.csv → Resuming...
Total rows: 9500
Rows already processed: 4671
Rows left to process: 4829



In [60]:
trainDF

,id,prompt,answer,cot_reasoning
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111,<think>\nHere's a thinking process that leads ...
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011,<think>\nThe user wants me to solve a puzzle b...
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book,<think>\nThe user wants me to explain the proc...
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII,<think>\nThe user wants me to identify the hid...
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret,<think>\nHere's a thinking process that leads ...
...,...,...,...,...
9495,ffce9e31,"In Alice's Wonderland, a secret bit manipulati...",01100110,NaN
9496,ffd5bada,"In Alice's Wonderland, a secret unit conversio...",32.45,NaN
9497,ffd89354,"In Alice's Wonderland, secret encryption rules...",student sees the curious mirror,NaN
9498,ffdfb678,"In Alice's Wonderland, secret encryption rules...",the curious mouse creates,NaN


In [61]:
%%time
if remaining == 0:
    print("✅ All rows already have CoT reasoning. Nothing to do.")
else:
    print("Starting CoT generation (resume mode)...\n")

    processed_count = 0

    for idx in tqdm(range(len(trainDF))):
        current_cot = trainDF.loc[idx, "cot_reasoning"]

        # Skip if already has content
        if pd.notna(current_cot) and str(current_cot).strip() != "":
            continue

        prompt = trainDF.loc[idx, "prompt"]
        answer = str(trainDF.loc[idx, "answer"]).strip()

        cot = generate_cot_data2(prompt, answer)
        trainDF.loc[idx, "cot_reasoning"] = cot
        processed_count += 1

        # Save progress every 50 new rows
        if processed_count % 20 == 0:
            trainDF.to_csv(outputFile, index=False)
            print(f"Saved progress. Processed {processed_count} new rows so far.")

        time.sleep(DELAY)

    #concurrent version:
    

    # Final save
    trainDF.to_csv(outputFile, index=False)
    print(f"\n✅ Finished! Processed {processed_count} new rows.")
    print(f"File saved to: {outputFile}")

Starting CoT generation (resume mode)...



 49%|█████████████████▊                  | 4691/9500 [07:27<12:15:47,  9.18s/it]

Saved progress. Processed 20 new rows so far.


 50%|█████████████████▊                  | 4711/9500 [14:50<27:56:41, 21.01s/it]

Saved progress. Processed 40 new rows so far.


...

 51%|█████████████████▌                | 4891/9500 [1:22:34<28:54:08, 22.58s/it]

Saved progress. Processed 220 new rows so far.


 52%|██████████████████                 | 4903/9500 [1:27:23<1:21:55,  1.07s/it]

CPU times: user 8.56 s, sys: 606 ms, total: 9.17 s
Wall time: 1h 27min 23s


KeyboardInterrupt: 

In [29]:
# Concurrent